# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:


# If your project provides these, keep them. If not, you can remove them.
# (They were shown in your starter as examples.)



In [3]:
# TODO: Import the necessary libs
import os
import re
import json
from typing import List, Dict, Any

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage
from lib.tooling import tool
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from tavily import TavilyClient


In [4]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
import os
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    #api_base="https://openai.vocareum.com/v1"
)
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

In [7]:
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

# TODO: Create retrieve_game tool
# It should use chroma client and collection you created

@tool
def retrieve_game(search: str, k: int = 5) -> List[Dict[str, Any]]:
    results = collection.query(
        query_texts=[search],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    games = []
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]

    for doc, meta, dist in zip(docs, metas, dists):
        meta = meta or {}
        games.append({
            "Platform": meta.get("Platform", "Unknown"),
            "Name": meta.get("Name", "Unknown"),
            "YearOfRelease": meta.get("YearOfRelease", "Unknown"),
            "Description": (meta.get("Description") or doc or "N/A"),
            "distance": float(dist) if dist is not None else None,
        })

    games.sort(key=lambda x: x["distance"] if x["distance"] is not None else 1e9)
    return games




#### Evaluate Retrieval Tool

In [8]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

In [9]:
import re

def _normalize(text: str) -> str:
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _tokens(text: str) -> set:
    stop = {
        "was","when","released","release","for","on","the","a","an","is","did","it","to","of",
        "game","version","platform","playstation","xbox","nintendo"
    }
    return {t for t in _normalize(text).split() if t not in stop and len(t) > 1}

def _overlap_ratio(q_tokens: set, name_tokens: set) -> float:
    if not q_tokens:
        return 0.0
    return len(q_tokens & name_tokens) / len(q_tokens)

from typing import Optional

@tool
def evaluate_retrieval(question: str, retrieved_docs: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    """
    Evaluate whether retrieved docs are sufficient.

    Args:
      - question: original user question
      - retrieved_docs: (optional) output from retrieve_game(). If missing, we retrieve inside.

    Returns dict:
      useful: bool
      description: str
      best_match_name: str|None
      best_match_distance: float|None
      token_overlap: float|None
      retrieved_count: int
    """
    # If the agent forgot to pass retrieved_docs, recover automatically
    if retrieved_docs is None:
        retrieved_docs = retrieve_game(question, k=5)

    if not retrieved_docs:
        return {
            "useful": False,
            "description": "No documents retrieved.",
            "best_match_name": None,
            "best_match_distance": None,
            "token_overlap": 0.0,
            "retrieved_count": 0,
        }

    q_tok = _tokens(question)

    scored = []
    for d in retrieved_docs:
        name = d.get("Name", "") or ""
        name_tok = _tokens(name)
        overlap = _overlap_ratio(q_tok, name_tok)
        dist = d.get("distance", None)
        scored.append((overlap, dist if dist is not None else 1e9, d))

    # best by overlap, then lowest distance
    scored.sort(key=lambda x: (-x[0], x[1]))
    best_overlap, _, best = scored[0]

    best_name = best.get("Name")
    best_dist = best.get("distance", None)

    # tighten/loosen as needed
    overlap_ok = best_overlap >= 0.60
    dist_ok = (best_dist is None) or (best_dist <= 0.6)

    useful = overlap_ok and dist_ok

    return {
        "useful": bool(useful),
        "description": (
            f"Internal retrieval matches question (overlap={best_overlap:.2f}, distance={best_dist})."
            if useful
            else f"Internal results do not match well enough (best overlap={best_overlap:.2f}, distance={best_dist}). Use web search."
        ),
        "best_match_name": best_name,
        "best_match_distance": best_dist,
        "token_overlap": float(best_overlap),
        "retrieved_count": len(retrieved_docs),
    }




#### Game Web Search Tool

In [10]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

In [11]:
# TODO: Create game_web_search tool
from tavily import TavilyClient
import urllib.parse

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> List[Dict[str, Any]]:
    """
    Web search fallback using Tavily.
    Returns list of {title,url,content,site_name}.
    """
    if not TAVILY_API_KEY:
        return []

    response = tavily_client.search(query=question, search_depth="basic", max_results=5)

    results = []
    for r in response.get("results", []):
        parsed = urllib.parse.urlparse(r.get("url", "") or "")
        site_name = parsed.netloc.replace("www.", "") if parsed.netloc else "Unknown Source"
        results.append({
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
            "site_name": site_name,
        })
    return results


### Agent

In [12]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [13]:
from lib.agents import Agent
from lib.llm import LLM

tools = [retrieve_game, evaluate_retrieval, game_web_search]

agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=(
        "You are UdaPlay, an AI Research Agent specialized in the video game industry.\n\n"
        "Workflow (MANDATORY):\n"
        "1) Call retrieve_game with the user's question.\n"
        "2) Call evaluate_retrieval(question=<user question>, retrieved_docs=<output of retrieve_game>)\n"
        "3) If payload evaluation says useful=false, call game_web_search(question=<user question>) and answer using web.\n"
        "4) If useful=true, answer using retrieved_docs.\n\n"
        "Hard rules:\n"
        "- Never answer from internal docs when useful=false.\n"
        "- Never guess.\n"
        "- If web search returns no results, say: \"I don't know\".\n\n"
        "Return JSON ONLY in this schema:\n"
        "{\n"
        '  \"answer\": \"...\",\n'
        '  \"source\": \"web search\" | \"internal knowledge\",\n'
        '  \"reasoning\": \"...\",\n'
        '  \"citations\": {\n'
        '    \"web_sources\": [[\"url\",\"site_name\"]],\n'
        '    \"local_sources\": [\"Platform: X, Game: Y, Year: Z\"]\n'
        '  }\n'
        "}\n"
    )
)



In [14]:
def log_tool_execution(messages):
    tool_calls = [msg for msg in messages if hasattr(msg, "role") and msg.role == "tool"]
    for tool_msg in tool_calls:
        tool_name = getattr(tool_msg, "name", "Unknown Tool")
        content = getattr(tool_msg, "content", "")
        print(f"Tool used: {tool_name}")
        print(f"Tool output (head): {str(content)[:200]}...")

def format_citations(answer_text):
    """Extract and format citations from structured JSON answer"""
    try:
        data = json.loads(answer_text)
    except Exception:
        return answer_text

    if isinstance(data, dict) and "citations" in data:
        print("\n=== CITATIONS ===")
        cites = data["citations"]

        if cites.get("web_sources"):
            print("Web Sources:")
            for url, site in cites["web_sources"]:
                print(f"  - {site}: {url}")

        if cites.get("local_sources"):
            print("Local Sources:")
            for s in cites["local_sources"]:
                print(f"  - {s}")

        print(f"Source Type: {data.get('source')}")
        print(f"Reasoning: {data.get('reasoning')}")
        print("================")

    return data.get("answer", answer_text)


In [15]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

In [16]:
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
    "When was EA Sports FC 25 released?"
]

for i, q in enumerate(queries, start=1):
    print(f"\n=== Query {i}: {q} ===")

    result = agent.invoke(query=q).get_final_state()

    # optional: show tool usage
    log_tool_execution(result["messages"])

    answer_content = result["messages"][-1].content
    print("ANSWER:", format_citations(answer_content))



=== Query 1: When Pokémon Gold and Silver was released? ===
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Tool used: retrieve_game
Tool output (head): "[{'Platform': 'Game Boy Color', 'Name': 'Pok\u00e9mon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay...
Tool used: evaluate_retrieval
Tool output (head): "{'useful': True, 'description': 'Internal retrieval matches question (overlap=1.00, distance=0.24596673250198364).', 'best_match_name': 'Pok\u00e9mon Gold and Silver', 'best_match_distance': 0.245966...

=== CITATIONS ===
Local Sources:
  - Platform: Game Boy Color, Game: Pokémon Gold and S

### (Optional) Advanced

In [17]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

In [23]:
# =========================
# Advanced: StateMachine + long-term memory
# Uses YOUR provided lib.state_machine implementation
# =========================

from typing import TypedDict, List, Dict, Any, Optional
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Resource

import chromadb
import json
from datetime import datetime

# -------------------------
# 1) Long-term memory (Chroma)
# -------------------------
memory_client = chromadb.PersistentClient(path="chromadb")
memory_collection = memory_client.get_or_create_collection(
    name="udaplay_long_term_memory",
    embedding_function=embedding_fn,  # reuse your embedding function
)

def _mem_pack(payload: Dict[str, Any]) -> str:
    return json.dumps(payload, ensure_ascii=False)

def memory_retrieve_local(query: str, k: int = 3) -> List[Dict[str, Any]]:
    res = memory_collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    out = []
    for doc, md, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        try:
            parsed = json.loads(doc)
        except Exception:
            parsed = {"raw": doc}
        out.append({
            "memory": parsed,
            "distance": float(dist) if dist is not None else None,
            "metadata": md or {},
        })
    out.sort(key=lambda x: x["distance"] if x["distance"] is not None else 1e9)
    return out

def memory_store_local(question: str, answer: str, citations: Dict[str, Any], source: str) -> Dict[str, Any]:
    payload = {
        "question": question,
        "answer": answer,
        "citations": citations,
        "source": source,
        "stored_at": datetime.utcnow().isoformat() + "Z",
    }
    doc_id = f"mem_{abs(hash(question + answer))}"
    memory_collection.upsert(
        ids=[doc_id],
        documents=[_mem_pack(payload)],
        metadatas=[{"type": "qa_memory", "source": source, "stored_at": payload["stored_at"]}],
    )
    return {"stored": True, "id": doc_id}


# -------------------------
# 2) Define State Schema (TypedDict)
# -------------------------
class UdaPlaySMState(TypedDict, total=False):
    # required by Agent.invoke initial_state
    user_query: str
    instructions: str
    messages: list
    session_id: str

    # state-machine fields we use
    memory_hits: List[Dict[str, Any]]
    use_memory: bool
    retrieved_docs: List[Dict[str, Any]]
    eval: Dict[str, Any]
    web_results: List[Dict[str, Any]]

    final: Dict[str, Any]
    should_store: bool
    store_payload: Dict[str, Any]
    tools_used: List[str]


# -------------------------
# 3) Step logic functions (each returns dict of updates)
# Note: Step.run will keep only fields present in UdaPlaySMState type hints.
# -------------------------

MEMORY_DISTANCE_THRESHOLD = 0.20  # tighten/loosen

def step_memory_lookup(state: UdaPlaySMState, resource: Resource=None) -> Dict[str, Any]:
    q = state["user_query"]
    hits = memory_retrieve_local(q, k=3)
    use_memory = bool(hits) and ((hits[0].get("distance") or 1e9) <= MEMORY_DISTANCE_THRESHOLD)

    tools_used = list(state.get("tools_used", []))
    tools_used.append("memory_retrieve")

    return {
        "memory_hits": hits,
        "use_memory": use_memory,
        "tools_used": tools_used,
    }

def step_retrieve(state: UdaPlaySMState, resource: Resource=None) -> Dict[str, Any]:
    q = state["user_query"]
    docs = retrieve_game(q, k=5)

    tools_used = list(state.get("tools_used", []))
    tools_used.append("retrieve_game")

    return {
        "retrieved_docs": docs,
        "tools_used": tools_used,
    }

def step_evaluate(state: UdaPlaySMState, resource: Resource=None) -> Dict[str, Any]:
    q = state["user_query"]
    docs = state.get("retrieved_docs", []) or []
    ev = evaluate_retrieval(q, docs)  # your fixed signature

    tools_used = list(state.get("tools_used", []))
    tools_used.append("evaluate_retrieval")

    return {
        "eval": ev,
        "tools_used": tools_used,
    }

def step_web_search(state: UdaPlaySMState, resource: Resource=None) -> Dict[str, Any]:
    q = state["user_query"]
    web = game_web_search(q)

    tools_used = list(state.get("tools_used", []))
    tools_used.append("game_web_search")

    return {
        "web_results": web,
        "tools_used": tools_used,
    }

def step_answer(state: UdaPlaySMState, resource: Resource=None) -> Dict[str, Any]:
    q = state["user_query"]

    # 1) memory path
    if state.get("use_memory"):
        top = (state.get("memory_hits") or [])[0]["memory"]
        final = {
            "answer": top.get("answer", "I don't know"),
            "source": "long-term memory",
            "reasoning": "Strong match found in long-term memory.",
            "citations": top.get("citations", {"web_sources": [], "local_sources": []}),
        }
        return {"final": final, "should_store": False}

    # 2) internal path
    ev = state.get("eval", {}) or {}
    if ev.get("useful"):
        docs = state.get("retrieved_docs", []) or []
        best = docs[0] if docs else {}

        local_sources = []
        if best:
            local_sources.append(
                f"Platform: {best.get('Platform')}, Game: {best.get('Name')}, Year: {best.get('YearOfRelease')}"
            )

        final = {
            "answer": (
                f"{best.get('Name')} was released in {best.get('YearOfRelease')} for {best.get('Platform')}. "
                f"{best.get('Description','')}"
            ).strip() if best else "I don't know",
            "source": "internal knowledge",
            "reasoning": ev.get("description", "Answered from internal retrieval."),
            "citations": {"web_sources": [], "local_sources": local_sources},
        }
        return {"final": final, "should_store": False}

    # 3) web path
    web = state.get("web_results", []) or []
    if not web:
        final = {
            "answer": "I don't know",
            "source": "web search",
            "reasoning": "Internal retrieval insufficient and web search returned no results.",
            "citations": {"web_sources": [], "local_sources": []},
        }
        return {"final": final, "should_store": False}

    top = web[0]
    url = top.get("url")
    site = top.get("site_name", "Unknown")

    final = {
        "answer": (top.get("content") or "").strip(),
        "source": "web search",
        "reasoning": ev.get("description", "Used web search fallback."),
        "citations": {"web_sources": [[url, site]], "local_sources": []},
    }
    store_payload = {
        "question": q,
        "answer": final["answer"],
        "citations": final["citations"],
        "source": "web search",
    }
    return {"final": final, "should_store": True, "store_payload": store_payload}

def step_store_memory(state: UdaPlaySMState, resource: Resource=None) -> Dict[str, Any]:
    if state.get("should_store") and state.get("store_payload"):
        p = state["store_payload"]
        memory_store_local(
            question=p["question"],
            answer=p["answer"],
            citations=p["citations"],
            source=p["source"],
        )

        tools_used = list(state.get("tools_used", []))
        tools_used.append("memory_store")

        return {"tools_used": tools_used}

    return {}


# -------------------------
# 4) Build StateMachine with your API
# -------------------------
sm = StateMachine(UdaPlaySMState)

entry = EntryPoint()
term = Termination()

s_memory = Step("memory_lookup", step_memory_lookup)
s_retrieve = Step("retrieve", step_retrieve)
s_eval = Step("evaluate", step_evaluate)
s_web = Step("web_search", step_web_search)
s_answer = Step("answer", step_answer)
s_store = Step("store_memory", step_store_memory)

sm.add_steps([entry, term, s_memory, s_retrieve, s_eval, s_web, s_answer, s_store])

# Connect graph
sm.connect(entry, s_memory)

# If memory hit is strong -> answer, else -> retrieve
sm.connect(s_memory, [s_answer, s_retrieve], condition=lambda st: "answer" if st.get("use_memory") else "retrieve")

sm.connect(s_retrieve, s_eval)

# If eval useful -> answer, else -> web_search
sm.connect(s_eval, [s_answer, s_web], condition=lambda st: "answer" if (st.get("eval", {}).get("useful")) else "web_search")

sm.connect(s_web, s_answer)

# If should_store -> store_memory else terminate
sm.connect(s_answer, [s_store, term], condition=lambda st: "store_memory" if st.get("should_store") else "__termination__")

sm.connect(s_store, term)


# -------------------------
# 5) Plug into Udacity Agent
# Agent uses self.workflow.run(initial_state)
# -------------------------
agent.workflow = sm


# -------------------------
# 6) Run (small output)
# -------------------------
def print_sm_result(run_obj):
    final_state = run_obj.get_final_state()
    final = (final_state or {}).get("final", {})
    tools_used = (final_state or {}).get("tools_used", [])
    print("Tools:", tools_used)
    print("Source:", final.get("source"))
    print("Answer:", (final.get("answer") or "")[:300], "...")
    print("Citations:", final.get("citations"))

queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
    "When was EA Sports FC 25 released?"
]

for i, q in enumerate(queries, start=1):
    print(f"\n=== Query {i}: {q} ===")
    # Use Agent.invoke to keep session memory; it calls agent.workflow.run(initial_state)
    run_obj = agent.invoke(query=q)
    print_sm_result(run_obj)



=== Query 1: When Pokémon Gold and Silver was released? ===
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_lookup
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__
Tools: ['memory_retrieve', 'retrieve_game', 'evaluate_retrieval']
Source: internal knowledge
Answer: Pokémon Gold and Silver was released in 1999 for Game Boy Color. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics. ...
Citations: {'web_sources': [], 'local_sources': ['Platform: Game Boy Color, Game: Pokémon Gold and Silver, Year: 1999']}

=== Query 2: Which one was the first 3D platformer Mario game? ===
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_lookup
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: web_search
[StateMachine] Executing step: answer
[StateMa

/tmp/ipykernel_18803/3000857214.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "stored_at": datetime.utcnow().isoformat() + "Z",


[StateMachine] Executing step: memory_lookup
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__
Tools: ['memory_retrieve']
Source: long-term memory
Answer: ***Mortal Kombat X*** is a 2015 fighting game developed by NetherRealm Studios and published by Warner Bros. An upgraded version of *Mortal Kombat X*, titled ***Mortal Kombat XL***, was released on March 1, 2016, for PlayStation 4 and Xbox One, including all downloadable content characters from the  ...
Citations: {'web_sources': [['https://en.wikipedia.org/wiki/Mortal_Kombat_X', 'en.wikipedia.org']], 'local_sources': []}

=== Query 4: When was EA Sports FC 25 released? ===
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_lookup
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__
Tools: ['memory_retrieve']
Source: long-term memory
Answer: EA Sports FC 25 will be released on Friday 27 September 2024 for every platform. If you buy the Ultimate Edition

In [ ]:
!tar cvfz UdaPlayAgent.tar.gz ./*